# Lesson 4b — Embedding + linear (first neural baseline)

Phase 2 of the Karpathy track: build PRAGMA incrementally. Same data and task throughout L4a-L4d, with one piece added each lesson.

Previous lesson result: **L4a counts: 63.4% acc, 0.670 CE**

Runnable version of `04b_*.py`.


## Setup — same data as L4a, plus PyTorch

In [ ]:
import random, torch
import torch.nn as nn

torch.manual_seed(0); random.seed(0)

KEYS   = ["pet", "action", "place"]
VALUES = ["dog", "cat", "fish", "eat", "sleep", "play", "garden", "couch", "bowl"]
PAD, MASK = "<pad>", "<mask>"
vocab  = [PAD, MASK] + KEYS + VALUES
tok2id = {t: i for i, t in enumerate(vocab)}
V = len(vocab)

RULES = {
    "dog":  {"action": ["eat", "play"],   "place": ["garden", "bowl"]},
    "cat":  {"action": ["sleep", "play"], "place": ["couch", "bowl"]},
    "fish": {"action": ["eat", "sleep"],  "place": ["bowl"]},
}

def random_event():
    pet = random.choice(list(RULES))
    act = random.choice(RULES[pet]["action"])
    plc = random.choice(RULES[pet]["place"])
    return {"pet": pet, "action": act, "place": plc}

def encode_event(ev):
    ids = []
    for k in KEYS:
        ids.append(tok2id[k])
        ids.append(tok2id[ev[k]])
    return ids

def make_mlm_example(ev, mask_field):
    ids = encode_event(ev)
    pos = 2 * KEYS.index(mask_field) + 1
    target_value_id = ids[pos]
    ids[pos] = tok2id[MASK]
    return ids, pos, target_value_id

def build_dataset(events):
    Xs, positions, ys = [], [], []
    for ev in events:
        for k in KEYS:
            ids, pos, tgt = make_mlm_example(ev, k)
            Xs.append(ids); positions.append(pos); ys.append(tgt)
    return (torch.tensor(Xs, dtype=torch.long),
            torch.tensor(positions, dtype=torch.long),
            torch.tensor(ys, dtype=torch.long))

train = [random_event() for _ in range(5000)]
test  = [random_event() for _ in range(1000)]
X_tr, pos_tr, y_tr = build_dataset(train)
X_te, pos_te, y_te = build_dataset(test)
print(f"X_tr: {tuple(X_tr.shape)}, y_tr: {tuple(y_tr.shape)}")

## Model: Embedding + mean-pool + Linear

The dumbest possible neural net. The embedding is LEARNED — that's where the power comes from.

In [ ]:
D = 16

class EmbLinearModel(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.emb  = nn.Embedding(V, d)
        self.head = nn.Linear(d, V)

    def forward(self, ids):
        h = self.emb(ids)            # (B, L, d)
        pooled = h.mean(dim=1)       # (B, d) — average across positions
        return self.head(pooled)     # (B, V)

model = EmbLinearModel(V, D)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

## Train

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

BATCH, N_STEPS = 128, 1000
for step in range(N_STEPS):
    idx = torch.randint(0, X_tr.size(0), (BATCH,))
    logits = model(X_tr[idx])
    loss = loss_fn(logits, y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0 or step == N_STEPS - 1:
        print(f"  step {step:4d}   loss {loss.item():.3f}")

## Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_te)
    pred = logits.argmax(-1)
    acc  = (pred == y_te).float().mean().item()
    ce   = loss_fn(logits, y_te).item()

print(f"  Test accuracy:       {acc * 100:.1f}%")
print(f"  Test cross-entropy:  {ce:.3f}")
print()
print(f"  L4a (counts):       63.4% acc,  0.670 CE")
print(f"  L4b (emb+linear):   {acc * 100:.1f}% acc,  {ce:.3f} CE")

## Next

[**L4c**](lesson_04c_with_attention.ipynb) — replace mean-pool with attention so the masked token can look at the visible ones.